# 资源分配问题

In [1136]:
import numpy as np
from time import time

## 类定义
定义node，表示节点类

In [1137]:
class node():
    def __init__(self, name=None, resourse=None):
        self.name = name
        self.resourse = resourse # 节点所用资源的列表
        self.fpga = None  # 节点被分配到的FPGA序号
        self.net = []     # 节点所属线网的序号列表

class net():
    def __init__(self, nodelist=None):
        self.nodelist = nodelist        # 线网中的节点的列表，此列表中每个元素是节点的名字
        self.fpga = set()    # 线网中的节点被分配到的FPGA的集合， 此集合中每个元素是FPGA的序号


## 定义函数
### 文件信息读取
读取文件，取得节点资源信息

In [1138]:
def readNodeFile(fileName):
    nodes = {}
    f = open(fileName)
    for line in f:
        name = line.split()[0]
        resources = sum(map(int, line.split()[-10:]))
        nodes[name] = node(name, resources)
    f.close()
    return nodes

读取文件，取得线网信息

In [1139]:
def readNetFile(filename):
    lineStack = []
    nodelist = []
    netId = 0
    nets = []
    f = open(filename)
    for line in f:
        lineStack.append(line.split())
    while lineStack != []:
        line = lineStack.pop()
        nodelist.append(line[0])
        if 's' in line:
            for name in nodelist:
                nodes[name].net.append(netId)
            nets.append(net(nodelist))
            nodelist = []
            netId += 1
    f.close()
    return nets


### 定义分配函数allocation

In [1140]:
def allocation():
    for node in nodes.values():
        node.fpga=np.random.randint(nosFPGA)
        for net in nets:
            if node.name in net.nodelist:
                net.fpga.add(node.fpga)
    return nodes


### 定义写文件函数writeResult

In [1141]:
def writeResult():
    for fpga in range(nosFPGA):
        f.write("F"+str(fpga))
        for node in nodes.values():
            if node.fpga==fpga:
                f.write(" "+node.name)
        f.write("\n")

### 定义资源差异函数 resourseCal
计算各个板的资源

In [1142]:
def resourseCal(nodes):
    resourses = [0]*nosFPGA
    for node in nodes.values():
        resourses[node.fpga]+=node.resourse
    return resourses

### 定义板间连线计算函数 linkCal
计算FPGA板间连线的总和

In [1143]:
def linkCal(nodes):
    link = 0
    # for n1 in nodes.values():
    #     for n2 in nodes.values():
    #         if n1.fpga != n2.fpga:
    #             link+= len(np.intersect1d(n1.net,n2.net))
    for net in nets:
        for name1 in net.nodelist:
            for name2 in net.nodelist:
                if nodes[name1].fpga == nodes[name2].fpga:
                    link += 1
    return int(link/2)


### 清空分配信息
清除节点中的分配信息

In [1144]:
def clearAssignInfo():
    for node in nodes.values():
        node.fpga = None

## 执行代码
读取文件

In [1145]:
# 读取节点文件
nodes = readNodeFile("./contest/testdata-0/design.are")

# 读取线网文件
nets = readNetFile("./contest/testdata-0/design.net")

设定参数

In [1146]:
nosFPGA = 4 
N = 10		# 染色体群体规模
MAXITER = 10**2  # 最大循环次数
pc = 0.15		# 交叉概率
pw = 0.85		# 变异概率

把结果文件打开

In [1147]:
f = open("./result.res", 'w')

第1种情况：

In [1148]:
clearAssignInfo()
allocation()
writeResult()
f.write(str(np.std(resourseCal(nodes)))+"\n")

18

第2种情况：

In [1149]:
# 产生N个染色体的初始群体,保存在pop里
# 每个染色体代表资源分配问题的某一个解（所有节点的FPGA分配情况）

def initPop():
    pop = []
    for i in range(N):
        pop.append(allocation())
    return pop

In [1150]:
# 计算pop中每个染色体所代表的轨迹的长度

def calLen():
    links = []
    for individ in pop:
        links.append(linkCal(individ))
    return links

In [1151]:
# 根据所有染色体分别代表衡量分配好坏的值，计算各个染色体的适应值fitness

def calFitness():
    fitness = []
    for link in links:
        fitness.append(1/link)
    return fitness

In [1152]:
# 根据染色体群体pop已经对应的适应值fitness，
# 找到最高的适应值f，以及对应的染色体bst和其在pop中的编号/下标ind

def findBest():
    maxFit = max(fitness)
    index = fitness.index(maxFit)
    best = pop[index]
    return [best, maxFit, index]

def satified(fitness):
    return 0

In [1153]:
# 根据染色体的适应值，按照一定的概率，生成新一代染色体群体newpop

def chooseNewP():
    N = len(pop)
    sumFitness = np.cumsum(fitness)
    newPop = []
    for _ in range(N):
        randVal = np.random.rand()
        for i in range(N-1, -1, -1):
            if randVal > sumFitness[i]:
                newPop.append(pop[i])
                break
    return newPop


In [1154]:
# 根据交叉概率pc，以及各染色体的适应值fitness，通过交叉的方式生成新群体
# #SelfAdj = 1时为自适应，否则取固定的交叉概率pc

def crossPop():
    N = len(pop)
    lst = list(range(N))
    for i in range(int(N/2)):
        randval = np.random.rand()
        j = np.random.randint(int(N/2), N)
        if randval < pc:
            part1 = np.copy(pop[lst[i]])
            part2 = np.copy(pop[lst[j]])
            if part1 != part2:
                child1, child2 = genecross(part1, part2)
                pop[lst[i]] = np.copy(child1)
                pop[lst[j]] = np.copy(child2)

In [1155]:
# 父染色体part1,part2，通过交叉方式
# 生成两个子染色体child1,child2

def genecross(part1, part2):
    indexs = np.random.sample(range(len(part1)), 2)
    index1 = max(indexs)
    index2 = min(indexs)
    child1 = np.copy(part1)
    child2 = np.copy(part2)
    temp1 = np.copy(child1[index1:index2])
    temp2 = np.copy(child2[index1:index2])
    if (temp1 == temp2).all():
        return [child1, child2]
    if set(temp1) == set(temp2):
        child1[index1:index2] = temp2
        child2[index1:index2] = temp1
        return [child1, child2]
    diff1 = set(temp1).difference(set(temp2))
    diff2 = set(temp2).difference(set(temp1))
    for i in range(len(diff1)):
        child1[np.where(child1 == list(diff2)[i])] = list(diff1)[i]
        child2[np.where(child2 == list(diff1)[i])] = list(diff2)[i]
    child1[index1:index2] = temp2
    child2[index1:index2] = temp1
    return [child1, child2]

In [1156]:

def mutPop():
    maxFit = max(fitness)
    meanFit = np.mean(fitness)
    for individ in pop:
        randval = np.random.random()
        if randval > pw:
            continue
			# 同上交叉操作，如果随机数大于概率就不进行变异操作，否者进行
        individ = np.asarray(mutLinkGene(individ))  # 将输入数据转为矩阵模式

# 随机选取一个位置进行变异


def mutLinkGene(node):
    index1, index2 = 0, 0
    while index1 == index2:
        index1 = np.random.randint(0, N-1)  # 随机选取某一个基因位置发生变异
        index2 = np.random.randint(0, N-1)
    node[index1], node[index2] = node[index2], node[index1]  # 交换位置，完成变异
    return node


In [1157]:
clearAssignInfo()

# 步骤1，产生N个染色体的初始群体,保存在pop里面
pop = initPop()  # pop 遗传算法中的种群

tstart = time()
for iter in range(MAXITER):
    links = calLen()  # 步骤2：计算每个染色体的路径长度
    fitness = calFitness()  # 计算每个染色体的适应值
    best, maxFit, index = findBest()			# 找到在当前种群中，适应度最高的个体

    # 步骤3：如果满足某些标准，算法停止
    if satified(fitness):
        break
    pop = chooseNewP()  # 否则，选取出一个新的群体
    crossPop()  # 步骤4：交叉产生新染色体，得到新群体
    # mutPop()  # 步骤5：基因变异
    pop[0] = best.copy()					# 保留上一代中适应值最高的染色体
    if np.mod(iter, MAXITER/10) == 1:
        tend = time()
        print("迭代次数： %d 运行时间： %f 秒 板间连线总数：%d" %
              (iter, tend-tstart, links[index]))

# 输出最优染色体/路径
print('最优连线数量：%f，适应值：%f，对应分配关系：' % (links[index], maxFit))
print(list(best))

# 计算运行时间
tend = time()
print('问题耗时 :  %f 秒.' % (tend-tstart))

writeResult()
f.write(str(linkCal(best))+"\n")


迭代次数： 1 运行时间： 0.012693 秒 板间连线总数：156
迭代次数： 11 运行时间： 0.076545 秒 板间连线总数：156
迭代次数： 21 运行时间： 0.136332 秒 板间连线总数：156
迭代次数： 31 运行时间： 0.190469 秒 板间连线总数：156
迭代次数： 41 运行时间： 0.241048 秒 板间连线总数：156
迭代次数： 51 运行时间： 0.286263 秒 板间连线总数：156
迭代次数： 61 运行时间： 0.323846 秒 板间连线总数：156
迭代次数： 71 运行时间： 0.359772 秒 板间连线总数：156
迭代次数： 81 运行时间： 0.391936 秒 板间连线总数：156
迭代次数： 91 运行时间： 0.423321 秒 板间连线总数：156
最优连线数量：156.000000，适应值：0.006410，对应分配关系：
['g0', 'g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9', 'g10', 'g11', 'g12', 'g13', 'g14', 'g15', 'g16', 'g17', 'g18', 'g19', 'g20', 'g21', 'g22', 'g23', 'g24', 'g25', 'g26', 'g27', 'g28', 'g29', 'g30', 'gp0', 'gp1', 'gp2', 'gp3', 'gp4', 'gp5', 'gp6', 'gp7', 'gp8', 'gp9', 'gp10', 'gp11', 'gp12', 'gp13', 'gp14', 'gp15', 'gp16', 'gp17', 'gp18', 'gp19', 'gp20', 'gp21']
问题耗时 :  0.444640 秒.


4

第3种情况：

In [1158]:
clearAssignInfo()
allocation()
writeResult()
f.write(str(linkCal(best))+"\n")

4

关闭文件

In [1159]:
f.close()

## 各种测试语句：

In [1160]:
print(linkCal())

TypeError: linkCal() missing 1 required positional argument: 'nodes'

In [ ]:
for i in range(nosFPGA):
    for node in nodes.values():
        if node.fpga==i:
            print(node.name)
            

g2
g6
g16
g17
g21
g27
g29
gp0
gp3
gp7
gp19
gp20
g1
g3
g4
g8
g15
g18
g19
g24
gp10
gp11
g0
g5
g7
g10
g13
g14
g20
g23
g25
g26
g28
g30
gp2
gp4
gp8
gp12
gp13
gp14
gp21
g9
g11
g12
g22
gp1
gp5
gp6
gp9
gp15
gp16
gp17
gp18


In [ ]:
print(resourseCal())

[102, 187, 184, 24]


In [ ]:
np.std([276, 100, 24, 97])

92.74797841462637

In [ ]:
sum([65, 6, 6, 3, 1, 1, 1, 1, 1])

85

In [ ]:
np.std([85,110,162,140])

29.22648627529488